In [ ]:
import numpy as np

class SevenWondersDecoder:
    def __init__(self, card_catalog, wonder_catalog, token_catalog):
        self.card_catalog = card_catalog
        self.wonder_catalog = wonder_catalog
        self.token_catalog = token_catalog
        self.num_cards = 73

    def decode(self, vector):
        res = []
        res.append("=== STAN GLOBALNY ===")
        res.append(f"Aktywny Gracz: {'Gracz 1' if vector[0] > 0.5 else 'Gracz 2'}")
        
        epoki = ["I", "II", "III"]
        epoka = next((epoki[i] for i in range(3) if vector[2+i] > 0.5), "Nieznana")
        res.append(f"Epoka: {epoka}")
        res.append(f"Pozycja Konfliktu: {vector[5] * 18 - 9:.0f}")
        res.append(f"Koniec Gry: {bool(vector[6])}")

        # Sekcje Graczy
        res.append(self._decode_player(vector, 48, "AKTYWNY GRACZ"))
        res.append(self._decode_player(vector, 169, "PRZECIWNIK"))

        # Piramida (od indeksu 290)
        res.append("\n=== PIRAMIDA ===")
        # slot_size = 4 + self.num_cards
        # start_idx = 290
        # for i in range(20):
        #     idx = start_idx + (i * slot_size)
        #     if vector[idx] > 0.5: # IsPresent
        #         status = "Dostępna" if vector[idx+2] > 0.5 else ("Zakryta" if vector[idx+1] > 0.5 else "Odkryta")
        #         koszt = vector[idx+3] * 20 # Zakładając MaxCoins = 20
                
        #         # Szukanie nazwy karty
        #         card_bits = vector[idx+4 : idx+4+self.num_cards]
        #         card_name = "Nieznana"
        #         if any(card_bits > 0.5):
        #             card_name = self.card_catalog[np.argmax(card_bits)]
                
        #         res.append(f"Slot {i:2d}: {card_name:20} | {status:10} | Koszt: {koszt:2.0f}")

        return "\n".join(res)

    def _decode_player(self, vector, start_idx, label):
        p = [f"\n--- {label} ---"]
        p.append(f"Monety: {vector[start_idx] * 100:.0f} | PZ: {vector[start_idx+1] * 100:.0f}")
        
        # Surowce (G, K, D, S, P)
        s = vector[start_idx+2 : start_idx+7]
        p.append(f"Surowce: Glina:{s[0]*10:.0f} Kamień:{s[1]*10:.0f} Drewno:{s[2]*10:.0f} Szkło:{s[3]*10:.0f} Papirus:{s[4]*10:.0f}")
        
        # Cuda (posiadane/zbudowane)
        w_start = start_idx + 13 + self.num_cards + 7 # Przesunięcie po surowcach, nauce i kartach
        # built_wonders = []
        # for i in range(len(self.wonder_catalog)):
        #     w_idx = w_start + (i * 2)
        #     if vector[w_idx] > 0.5: # Owned
        #         status = "[ZBUDOWANO]" if vector[w_idx+1] > 0.5 else "[W PULI]"
        #         built_wonders.append(f"{self.wonder_catalog[i]} {status}")
        # p.append("Cuda: " + (", ".join(built_wonders) if built_wonders else "Brak"))
        
        return "\n".join(p)

# PRZYKŁAD UŻYCIA
# Musisz podać listy nazw w tej samej kolejności co w C#
cards = [
            "Wycinka",
            "Zloza Gliny",
            "Skladowisko Kamienia",
            "Glinianka",
            "Sklad Drewna",
            "Kamieniolom",
            "Huta Szkla",
            "Wytwornia Papirusu",
            "Tawerna",
            "Magazyn Kamienia",
            "Magazyn Drewna",
            "Magazyn Gliny",
            "Skryptorium",
            "Apteka",
            "Zielarnia",
            "Warsztat",
            "Garnizon",
            "Palisada",
            "Wieza Straznicza",
            "Stajnie",
            "Teatr",
            "Laznie",
            "Oltarz",
            "Kamieniolom Stokowy",
            "Cegielnia",
            "Tartak",
            "Pracownia Wyrobu Szkla",
            "Suszarnia Papirusu",
            "Karawanseraj",
            "Urzad celny",
            "Browar",
            "Forum",
            "Labolatorium",
            "Biblioteka",
            "Szkola",
            "Ambulatorium",
            "Mury Obronne",
            "Plac Apelowy",
            "Koszary",
            "Stadniny",
            "Tor Strzelecki",
            "Gmach Sadu",
            "Akwedukt",
            "Mownica",
            "Swiatynia",
            "Posag",
            "Latarnia Morska",
            "Port",
            "Arena",
            "Izba Handlowa",
            "Zbrojownia",
            "Akademia",
            "Pracownia Naukowa",
            "Uniwersytet",
            "Obserwatorium",
            "Cyrk",
            "Fortyfikacje",
            "Arsenal",
            "Siedziba Trybuna",
            "Machiny Obleznicze",
            "Obelisk",
            "Budynek Senatu",
            "Ratusz",
            "Palac",
            "Panteon",
            "Ogrody",
            "Cech Lichwiarzy",
            "Cech Budowniczych",
            "Gildia Kupiecka",
            "Stowarzyszenie Urzednikow",
            "Cech Armatorow",
            "Gildia Strategow",
            "Towarzystwo Naukowe"
          ] # Pełna lista 73 kart
wonders = ["Piramidy", "Wiszące Ogrody", "..."] # 12 cudów
tokens = ["Rolnictwo", "Filozofia", "Prawo", "Strategia", "Matematyka", 
          "Architektura", "Budownictwo", "Urbanistyka", "Ekonomia", "Teologia"] # 10 żetonów

decoder = SevenWondersDecoder(cards, wonders, tokens)

In [ ]:
import glob
import os

list_of_files = glob.glob('../../GameTests/EncoderResults/*.txt') 
# list_of_files = glob.glob('C:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders/GameTests/EncoderResults/*.txt')
for file in list_of_files:    
    print(f"Znaleziono plik: {file}")
    raw_vector = np.loadtxt(file)
    print(decoder.decode(raw_vector))

Znaleziono plik: ../../GameTests/EncoderResults\vector_state.txt
=== STAN GLOBALNY ===
Aktywny Gracz: Gracz 1
Epoka: I
Pozycja Konfliktu: 0
Koniec Gry: False

--- AKTYWNY GRACZ ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

--- PRZECIWNIK ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

=== PIRAMIDA ===
Znaleziono plik: ../../GameTests/EncoderResults\vector_state_after_move.txt
=== STAN GLOBALNY ===
Aktywny Gracz: Gracz 1
Epoka: I
Pozycja Konfliktu: 0
Koniec Gry: False

--- AKTYWNY GRACZ ---
Monety: 6 | PZ: 0
Surowce: Glina:0 Kamień:1 Drewno:0 Szkło:0 Papirus:0

--- PRZECIWNIK ---
Monety: 7 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

=== PIRAMIDA ===


In [ ]:
text = decoder.decode(raw_vector)
print(text)

=== STAN GLOBALNY ===
Aktywny Gracz: Gracz 1
Epoka: II
Pozycja Konfliktu: 0
Koniec Gry: False

--- AKTYWNY GRACZ ---
Monety: 0 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

--- PRZECIWNIK ---
Monety: 0 | PZ: 0
Surowce: Glina:0 Kamień:0 Drewno:0 Szkło:0 Papirus:0

=== PIRAMIDA ===


In [1]:
import glob
import os
from game_state_decoder import BatchedGameStateDecoder
import numpy as np
decoder = BatchedGameStateDecoder()
# path = glob.glob('../../GameConsole/Results/self_play_puct_20260519_141051_1000_games_minimal.npz')
path = glob.glob('../../GameConsole/Results/mcts_1000_games.npz')
# path = glob.glob('training_data_3.npz')
states, actions, masks = decoder.load_and_preprocess(path[0], normalize=True)

states.shape, actions.shape, masks.shape

BadZipFile: File is not a zip file

In [2]:
# wypisz te gdzie states ma wartość 1.0
states, actions, masks = decoder.load_and_preprocess(path[0], normalize=False)

print("raw min/max/mean/std:", states.min(), states.max(), states.mean(), states.std())

# kilka prostych invariants
print("active player one-hot sum:", states[:, :2].sum(axis=1)[:5])
print("epoch one-hot sum:", states[:, 2:5].sum(axis=1)[:5])
print("action mask unique values:", np.unique(masks))
print("nonzero per state:", np.count_nonzero(states, axis=1)[:5])

raw min/max/mean/std: 0.0 1.0 0.04771839 0.20876597
active player one-hot sum: [1. 1. 1. 1. 1.]
epoch one-hot sum: [1. 1. 1. 1. 1.]
action mask unique values: [0. 1.]
nonzero per state: [84 83 83 83 84]


In [3]:
# Diagnostic check for encoded training data
import numpy as np

npz_path = path[0] if isinstance(path, list) else path

raw_states, raw_actions, raw_masks = decoder.load_and_preprocess(npz_path, normalize=False)
normalized_states, _, _ = decoder.load_and_preprocess(npz_path, normalize=True)

print(f"NPZ path: {npz_path}")
print(f"Raw states shape: {raw_states.shape}")
print(f"Actions shape: {raw_actions.shape}")
print(f"Masks shape: {raw_masks.shape}")
print()
print("Raw state stats:")
print(f"  Min: {raw_states.min():.4f}, Max: {raw_states.max():.4f}")
print(f"  Mean: {raw_states.mean():.4f}, Std: {raw_states.std():.4f}")
print(f"  Nonzero per state (first 10): {np.count_nonzero(raw_states, axis=1)[:10]}")
print()
print("Invariants:")
print(f"  Active player one-hot sums (first 5): {raw_states[:, :2].sum(axis=1)[:5]}")
print(f"  Epoch one-hot sums (first 5): {raw_states[:, 2:5].sum(axis=1)[:5]}")
print(f"  Action mask unique values: {np.unique(raw_masks)}")
print(f"  Actions range: {raw_actions.min()}..{raw_actions.max()}")
print(f"  States contain zeros: {bool(np.any(raw_states == 0.0))}")
print(f"  States contain ones: {bool(np.any(raw_states == 1.0))}")
print()
print("Normalized state stats:")
print(f"  Min: {normalized_states.min():.4f}, Max: {normalized_states.max():.4f}")
print(f"  Mean: {normalized_states.mean():.4f}, Std: {normalized_states.std():.4f}")
print()
print("Quick sanity verdict:")
if raw_states.shape[1] != decoder._calculate_state_dimension():
    print("  ❌ State dimension mismatch")
elif not np.allclose(raw_states[:, :2].sum(axis=1), 1.0):
    print("  ❌ Active player one-hot is invalid")
elif not np.allclose(raw_states[:, 2:5].sum(axis=1), 1.0):
    print("  ❌ Epoch one-hot is invalid")
elif not np.array_equal(np.unique(raw_masks), np.array([0.0, 1.0])):
    print("  ❌ Action mask is not binary")
elif np.count_nonzero(raw_states) == 0:
    print("  ❌ States are empty")
else:
    print("  ✅ Basic structural checks passed")

NPZ path: ../../GameConsole/Results/self_play_puct_20260519_141051_1000_games_minimal.npz
Raw states shape: (58448, 1903)
Actions shape: (58448,)
Masks shape: (58448, 120)

Raw state stats:
  Min: 0.0000, Max: 1.0000
  Mean: 0.0477, Std: 0.2088
  Nonzero per state (first 10): [84 83 83 83 84 87 89 88 85 87]

Invariants:
  Active player one-hot sums (first 5): [1. 1. 1. 1. 1.]
  Epoch one-hot sums (first 5): [1. 1. 1. 1. 1.]
  Action mask unique values: [0. 1.]
  Actions range: 0..119
  States contain zeros: True
  States contain ones: True

Normalized state stats:
  Min: -3.5557, Max: 9.6882
  Mean: 0.0405, Std: 0.2526

Quick sanity verdict:
  ✅ Basic structural checks passed


In [4]:
# Detailed breakdown of a few decoded states
npz_path = path[0] if isinstance(path, list) else path
sample_states, sample_actions, sample_masks = decoder.load_and_preprocess(npz_path, normalize=False)

state_dim = decoder._calculate_state_dimension()
continuous_dim = decoder._calculate_continuous_features()
card_offset = continuous_dim
wonder_offset = card_offset + 2 * decoder.NUM_CARDS
token_offset = wonder_offset + 4 * decoder.NUM_WONDERS
pyramid_offset = token_offset + 2 * decoder.NUM_PROGRESS_TOKENS
slot_block = 4 + decoder.NUM_CARDS
pyramid_size = decoder.MAX_BOARD_SLOTS * slot_block
discard_offset = pyramid_offset + pyramid_size

print(f"State dim: {state_dim}")
print(f"Continuous dim: {continuous_dim}")
print(f"Card offset: {card_offset}")
print(f"Wonder offset: {wonder_offset}")
print(f"Token offset: {token_offset}")
print(f"Pyramid offset: {pyramid_offset}")
print(f"Discard offset: {discard_offset}")
print()

# for idx in range(min(3, len(sample_states))):
for idx in range(60):
    state = sample_states[idx]
    print(f"=== Sample state {idx} ===")
    print(f"Action: {sample_actions[idx]}")
    print(f"Active player one-hot: {state[:2]}")
    print(f"Epoch one-hot: {state[2:5]}")
    print(f"Conflict position: {state[5]:.2f}, End game: {bool(state[6])}")
    # Brak, Militarne, Naukowe, Punktowe
    print(f"Typ zwyciestwa - Brak:{state[7]} Militarne:{state[8]} Naukowe:{state[9]} Punktowe:{state[10]}")
    # print(f"Indeksy stref 3x9 (CzyJuzUzyta, LiczbaTraconychMonet, LiczbaPunktow): {state[11:38]}")
    print(f"Indeksy zetonow postepu: {state[38:48]}")
    print(f"Player 1 cards nonzero count: {int(np.count_nonzero(state[card_offset:card_offset + decoder.NUM_CARDS]))}")
    print(f"Player 2 cards nonzero count: {int(np.count_nonzero(state[card_offset + decoder.NUM_CARDS:card_offset + 2 * decoder.NUM_CARDS]))}")
    print(f"Wonder block nonzero count: {int(np.count_nonzero(state[wonder_offset:wonder_offset + 4 * decoder.NUM_WONDERS]))}")
    print(f"Token block nonzero count: {int(np.count_nonzero(state[token_offset:token_offset + 2 * decoder.NUM_PROGRESS_TOKENS]))}")
    print(f"Pyramid block nonzero count: {int(np.count_nonzero(state[pyramid_offset:pyramid_offset + pyramid_size]))}")
    print(f"Discard block nonzero count: {int(np.count_nonzero(state[discard_offset:discard_offset + decoder.NUM_CARDS]))}")
    print(f"First 40 values: {state[:40]}")
    print()

State dim: 1903
Continuous dim: 76
Card offset: 76
Wonder offset: 222
Token offset: 270
Pyramid offset: 290
Discard offset: 1830

=== Sample state 0 ===
Action: 24
Active player one-hot: [1. 0.]
Epoch one-hot: [1. 0. 0.]
Conflict position: 0.50, End game: False
Typ zwyciestwa - Brak:1.0 Militarne:0.0 Naukowe:0.0 Punktowe:0.0
Indeksy zetonow postepu: [0. 1. 1. 0. 1. 0. 0. 0. 1. 1.]
Player 1 cards nonzero count: 3
Player 2 cards nonzero count: 2
Wonder block nonzero count: 2
Token block nonzero count: 2
Pyramid block nonzero count: 52
Discard block nonzero count: 0
First 40 values: [1.  0.  1.  0.  0.  0.5 0.  1.  0.  0.  0.  0.  0.  1.  0.  0.5 1.  0.
 0.2 0.5 0.  0.  0.2 1.  0.  0.  0.  0.  0.2 0.  0.2 0.5 0.  0.5 1.  0.
 0.  1.  0.  1. ]

=== Sample state 1 ===
Action: 0
Active player one-hot: [0. 1.]
Epoch one-hot: [1. 0. 0.]
Conflict position: 0.50, End game: False
Typ zwyciestwa - Brak:1.0 Militarne:0.0 Naukowe:0.0 Punktowe:0.0
Indeksy zetonow postepu: [0. 1. 1. 0. 1. 0. 0. 0. 1. 1